In [ ]:
import os
import csv
import threading
import time
import re
from datetime import datetime
from typing import List, Dict, Tuple

from flask import Flask, request, jsonify, send_from_directory
from flask_cors import CORS
from werkzeug.utils import secure_filename

# AI 相關庫
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from deepface import DeepFace

# ==============================================
# 0. 路徑與基礎設定 (指向 492G 大硬碟)
# ==============================================
BASE_DATA_PATH = "/mnt/data/f2227660_work"
DESKTOP_PATH = os.path.join(BASE_DATA_PATH, "小花聊天紀錄")
UPLOAD_DIR = os.path.join(BASE_DATA_PATH, "uploads")

os.makedirs(DESKTOP_PATH, exist_ok=True)
os.makedirs(UPLOAD_DIR, exist_ok=True)

app = Flask(__name__)
CORS(app)
app.config['MAX_CONTENT_LENGTH'] = 15 * 1024 * 1024  # 15MB 支援照片上傳

# ==============================================
# 1. CSV 記錄工具
# ==============================================
class DailyCSVLogger:
    def __init__(self, basename: str, fieldnames: List[str]):
        self.basename = basename
        self.fieldnames = fieldnames
        self.lock = threading.Lock()

    def path_for_today(self) -> str:
        day = datetime.now().strftime("%Y-%m-%d")
        return os.path.join(DESKTOP_PATH, f"{self.basename}_{day}.csv")

    def append(self, row: Dict):
        path = self.path_for_today()
        with self.lock:
            file_exists = os.path.exists(path)
            with open(path, "a", newline="", encoding="utf-8-sig") as f:
                writer = csv.DictWriter(f, fieldnames=self.fieldnames, extrasaction="ignore")
                if not file_exists:
                    writer.writeheader()
                writer.writerow(row)


CHAT_FIELDS = ["對話時間", "使用者ID", "小花名稱", "設定性格", "偵測情緒", 
               "使用者提問", "AI回覆", "長者姓名", "長者年齡", "健康備註"]
CHAT_LOGGER = DailyCSVLogger("chatlogs", CHAT_FIELDS)

# ==============================================
# 2. 性格與模型設定
# ==============================================
PERSONALITY_PROMPTS = {
    "溫暖貼心": "你非常溫柔，常關心長輩身體，會說『要記得喝水喔』，語氣充滿愛心，喜歡用❤️。",
    "活潑調皮": "你充滿活力，說話快，常說『快跟我玩！』或『汪汪！去散步吧！』，喜歡用🤩。",
    "可愛粘人": "你很黏人，喜歡用口頭禪如『汪汪』、『呀』，喜歡用表情🥺，想纏著主人一起做事。"
}

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# ==============================================
# 3. DeepFace 情緒分析
# ==============================================
def analyze_emotion_deepface(image_path: str) -> Tuple[str, float]:
    """使用 DeepFace 分析照片情緒"""
    try:
        results = DeepFace.analyze(
            img_path=image_path,
            actions=['emotion'],
            enforce_detection=False,
            detector_backend='opencv'
        )
        
        analysis = results[0] if isinstance(results, list) else results
        dominant_emotion = analysis['dominant_emotion']
        score = analysis['emotion'][dominant_emotion]

        emotion_zh = {
            "happy": "開心", "sad": "難過", "angry": "生氣",
            "surprise": "驚訝", "fear": "害怕", "disgust": "厭惡", "neutral": "平靜"
        }
        
        return emotion_zh.get(dominant_emotion.lower(), dominant_emotion), float(score)
    
    except Exception as e:
        print(f"❌ DeepFace 辨識出錯: {e}")
        return "平靜", 0.0


# ==============================================
# 4. API 路由
# ==============================================

# 聊天 API
@app.route('/ask_puppy', methods=['POST'])
def ask_puppy():
    try:
        data = request.json or {}
        user_msg = data.get('message', "")
        profile = data.get('user_profile') or {}
        
        pet_name = data.get('pet_name', "小花")
        personality = data.get('personality', "溫暖貼心")
        
        # --- 核心修正：初始化變數，避免 'not defined' 錯誤 ---
        detected_emotion = data.get('detected_emotion') 
        emotion_score = data.get('emotion_score', 0.0) # 預設為 0.0
        
        # 如果訊息中有【系統提示】，嘗試解析裡面的情緒文字
        if "【系統提示" in user_msg:
            match = re.search(r"情緒是「(.*?)」", user_msg)
            if match:
                detected_emotion = match.group(1)
        
        # 如果最後還是沒抓到情緒，給個預設值
        if not detected_emotion:
            detected_emotion = "平靜"
            
        # 在 ask_puppy 路由內修改 print 段落
        score_display = f"{emotion_score:.2f}" if emotion_score > 0 else "N/A (文字解析)"

        print(f"\n📥 收到使用者的訊息: {user_msg}")
        print(f"   偵測情緒: {detected_emotion}, confidence level: {score_display}")

        system_msg = (
            f"你現在是一隻名叫「{pet_name}」的虛擬寵物狗。你的主人是「{profile.get('name', '長輩')}」。"
            f"性格設定：{PERSONALITY_PROMPTS.get(personality)}。"
            f"【重要資訊】：主人現在看起來感覺「{detected_emotion}」。"
            f"請以小狗視角簡短回覆（30字內），要根據主人當下的情緒給予回應，絕對不要說自己是AI。"
        )

        messages = [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_msg}
        ]

        # Qwen 模型推論
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
        generated_ids = model.generate(**model_inputs, max_new_tokens=128, temperature=0.7)
        generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
        reply = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

        print(f"📤 小花回覆: {reply}\n")

        # 寫入 CSV
        CHAT_LOGGER.append({
            "對話時間": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "使用者ID": data.get('user_id', 'unknown'),
            "小花名稱": pet_name,
            "設定性格": personality,
            "偵測情緒": detected_emotion,
            "使用者提問": user_msg,
            "AI回覆": reply,
            "長者姓名": profile.get('name', '未填'),
            "長者年齡": profile.get('age', '未填'),
            "健康備註": profile.get('health_notes', '無')
        })

        return jsonify({"reply": reply, "detected_emotion": detected_emotion})

    except Exception as e:
        print(f"❌ 聊天發生錯誤: {e}")
        return jsonify({"reply": "汪汪...小花頭暈暈的🥺"}), 200


# 照片上傳 + DeepFace 情緒分析
@app.route('/checkin_photo', methods=['POST'])
def checkin_photo():
    try:
        file = request.files.get('photo')
        if not file:
            return jsonify({"message": "找不到照片"}), 400

        # 儲存照片
        filename = secure_filename(f"checkin_{int(time.time())}.jpg")
        save_path = os.path.join(UPLOAD_DIR, filename)
        file.save(save_path)

        # DeepFace 分析
        emotion, score = analyze_emotion_deepface(save_path)

        print(f"✅ 照片分析成功 | 情緒: {emotion} (信心度: {score:.2f}) | 檔案: {filename}")

        return jsonify({
            "emotion": emotion,
            "score": score,
            "file": filename,
            "message": f"主人看起來很{emotion}喔！"
        })

    except Exception as e:
        print(f"❌ 照片處理失敗: {e}")
        return jsonify({"message": "伺服器照片處理出錯"}), 500


# 下載聊天記錄
@app.route('/download_logs')
def download_logs():
    day = datetime.now().strftime("%Y-%m-%d")
    filename = f"chatlogs_{day}.csv"
    return send_from_directory(DESKTOP_PATH, filename, as_attachment=True)


# ==============================================
# 5. 啟動伺服器
# ==============================================
if __name__ == '__main__':
    # 👇 新增這兩行來強制隱藏沒有顯卡造成的警告
    os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
    os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
    
    print("\n🐾 小花伺服器啟動中 (DeepFace + Qwen 模式)...")
    
    # 預載入 Qwen 模型
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype="auto", device_map="auto")
    
    print("🧠 DeepFace 引擎初始化完成（第一次會自動下載模型）")
    print(f"✅ 伺服器啟動成功！數據路徑: {BASE_DATA_PATH}")
    
    app.run(host='0.0.0.0', port=5000, debug=False)

In [ ]:
import os
import csv
import threading
import time
import re
from datetime import datetime
from typing import List, Dict, Tuple

from flask import Flask, request, jsonify, send_from_directory
from flask_cors import CORS
from werkzeug.utils import secure_filename

# AI 相關庫
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from deepface import DeepFace

# ==============================================
# 0. 路徑與基礎設定 (指向 492G 大硬碟)
# ==============================================
BASE_DATA_PATH = "/mnt/data/f2227660_work"
DESKTOP_PATH = os.path.join(BASE_DATA_PATH, "小花聊天紀錄")
UPLOAD_DIR = os.path.join(BASE_DATA_PATH, "uploads")

os.makedirs(DESKTOP_PATH, exist_ok=True)
os.makedirs(UPLOAD_DIR, exist_ok=True)

app = Flask(__name__)
CORS(app)
app.config['MAX_CONTENT_LENGTH'] = 15 * 1024 * 1024  # 15MB 支援照片上傳

# ==============================================
# 1. CSV 記錄工具
# ==============================================
class DailyCSVLogger:
    def __init__(self, basename: str, fieldnames: List[str]):
        self.basename = basename
        self.fieldnames = fieldnames
        self.lock = threading.Lock()

    def path_for_today(self) -> str:
        day = datetime.now().strftime("%Y-%m-%d")
        return os.path.join(DESKTOP_PATH, f"{self.basename}_{day}.csv")

    def append(self, row: Dict):
        path = self.path_for_today()
        with self.lock:
            file_exists = os.path.exists(path)
            with open(path, "a", newline="", encoding="utf-8-sig") as f:
                writer = csv.DictWriter(f, fieldnames=self.fieldnames, extrasaction="ignore")
                if not file_exists:
                    writer.writeheader()
                writer.writerow(row)


CHAT_FIELDS = ["對話時間", "使用者ID", "小花名稱", "設定性格", "偵測情緒", 
               "使用者提問", "AI回覆", "長者姓名", "長者年齡", "健康備註"]
CHAT_LOGGER = DailyCSVLogger("chatlogs", CHAT_FIELDS)

# ==============================================
# 2. 性格與模型設定
# ==============================================
PERSONALITY_PROMPTS = {
    "溫暖貼心": "你非常溫柔，常關心長輩身體，會說『要記得喝水喔』，語氣充滿愛心，喜歡用❤️。",
    "活潑調皮": "你充滿活力，說話快，常說『快跟我玩！』或『汪汪！去散步吧！』，喜歡用🤩。",
    "可愛粘人": "你很黏人，喜歡用口頭禪如『汪汪』、『呀』，喜歡用表情🥺，想纏著主人一起做事。"
}

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# ==============================================
# 3. DeepFace 情緒分析
# ==============================================
def analyze_emotion_deepface(image_path: str) -> Tuple[str, float]:
    """使用 DeepFace 分析照片情緒"""
    try:
        results = DeepFace.analyze(
            img_path=image_path,
            actions=['emotion'],
            enforce_detection=False,
            detector_backend='opencv'
        )
        
        analysis = results[0] if isinstance(results, list) else results
        dominant_emotion = analysis['dominant_emotion']
        score = analysis['emotion'][dominant_emotion]

        emotion_zh = {
            "happy": "開心", "sad": "難過", "angry": "生氣",
            "surprise": "驚訝", "fear": "害怕", "disgust": "厭惡", "neutral": "平靜"
        }
        
        return emotion_zh.get(dominant_emotion.lower(), dominant_emotion), float(score)
    
    except Exception as e:
        print(f"❌ DeepFace 辨識出錯: {e}")
        return None, 0.0


# ==============================================
# 4. API 路由
# ==============================================

# 聊天 API
@app.route('/ask_puppy', methods=['POST'])
def ask_puppy():
    try:
        data = request.json or {}
        user_msg = data.get('message', "")
        profile = data.get('user_profile') or {}
        
        pet_name = data.get('pet_name', "小花")
        personality = data.get('personality', "溫暖貼心")
        
        # --- 核心修正：初始化變數，避免 'not defined' 錯誤 ---
        detected_emotion = data.get('detected_emotion') 
        emotion_score = data.get('emotion_score', 0.0) # 預設為 0.0
        
        
        # 如果訊息中有【系統提示】，嘗試解析裡面的情緒文字
        match = re.search(r"(?:情緒|心情)是[「『]?([^「『」』。！？\s]+)", user_msg)
        
        if match:
            detected_emotion = match.group(1).strip()
        
        # 如果最後還是沒抓到情緒，給個預設值
        if not detected_emotion or detected_emotion == "None":
            display_emotion = "--"
            emotion_context = "目前不確定主人的心情，請用平常心關心他。"
            
        else:
            display_emotion = detected_emotion
            emotion_context = f"【重要資訊】：主人現在看起來感覺「{detected_emotion}」，請針對這個情緒給予安慰或分享快樂。"
            
            
        # 在 ask_puppy 路由內修改 print 段落
        score_display = f"{emotion_score:.2f}" if emotion_score > 0 else "N/A (文字解析)"

        print(f"\n📥 收到使用者的訊息: {user_msg}")
        print(f"   偵測情緒: {detected_emotion}, confidence level: {score_display}")

        system_msg = (
            f"1. 你的身份：你是一隻叫「{pet_name}」的可愛虛擬狗狗。你的主人是「{profile.get('name', '長輩')}」。\n"
            f"2. 你的對象：現在正在跟你說話的人就是你的主人。他是一位「人類」，不是狗狗。\n"
            f"3. 你的性格：{PERSONALITY_PROMPTS.get(personality, '')}\n"
            f"4. 當前環境：主人現在心情看起來「{detected_emotion if detected_emotion else '平靜'}」。\n"
            f"5. 對話規則：\n"
            f"   - 必須用「第二人稱」直接跟主人說話（使用「你」或「主人」，禁止用第三人稱稱呼主人）。\n"
            f"   - 記住主人是人類，若提到食物，請推薦「人類的食物」，絕對不要叫主人吃狗糧。\n"
            f"   - 語氣要像狗狗，簡短溫暖（30字內），禁止提到你是AI、機器人或程式。"
        )

        messages = [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_msg}
        ]

        # Qwen 模型推論
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
        generated_ids = model.generate(**model_inputs, max_new_tokens=128, temperature=0.7)
        generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
        reply = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

        print(f"📤 小花回覆: {reply}\n")

        # 寫入 CSV
        CHAT_LOGGER.append({
            "對話時間": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "使用者ID": data.get('user_id', 'unknown'),
            "小花名稱": pet_name,
            "設定性格": personality,
            "偵測情緒": detected_emotion,
            "使用者提問": user_msg,
            "AI回覆": reply,
            "長者姓名": profile.get('name', '未填'),
            "長者年齡": profile.get('age', '未填'),
            "健康備註": profile.get('health_notes', '無')
        })

        return jsonify({"reply": reply, "detected_emotion": detected_emotion})

    except Exception as e:
        print(f"❌ 聊天發生錯誤: {e}")
        return jsonify({"reply": "汪汪...小花頭暈暈的🥺"}), 200


# 照片上傳 + DeepFace 情緒分析
@app.route('/checkin_photo', methods=['POST'])
def checkin_photo():
    try:
        file = request.files.get('photo')
        if not file:
            return jsonify({"message": "找不到照片"}), 400

        # 儲存照片
        filename = secure_filename(f"checkin_{int(time.time())}.jpg")
        save_path = os.path.join(UPLOAD_DIR, filename)
        file.save(save_path)

        # DeepFace 分析
        emotion, score = analyze_emotion_deepface(save_path)

        print(f"✅ 照片分析成功 | 情緒: {emotion} (信心度: {score:.2f}) | 檔案: {filename}")

        return jsonify({
            "emotion": emotion,
            "score": score,
            "file": filename,
            "message": f"主人看起來很{emotion}喔！"
        })

    except Exception as e:
        print(f"❌ 照片處理失敗: {e}")
        return jsonify({"message": "伺服器照片處理出錯"}), 500


# 下載聊天記錄
@app.route('/download_logs')
def download_logs():
    day = datetime.now().strftime("%Y-%m-%d")
    filename = f"chatlogs_{day}.csv"
    return send_from_directory(DESKTOP_PATH, filename, as_attachment=True)


# ==============================================
# 5. 啟動伺服器
# ==============================================
if __name__ == '__main__':
    # 👇 新增這兩行來強制隱藏沒有顯卡造成的警告
    os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
    os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
    
    print("\n🐾 小花伺服器啟動中 (DeepFace + Qwen 模式)...")
    
    # 預載入 Qwen 模型
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype="auto", device_map="auto")
    
    print("🧠 DeepFace 引擎初始化完成（第一次會自動下載模型）")
    print(f"✅ 伺服器啟動成功！數據路徑: {BASE_DATA_PATH}")
    
    app.run(host='0.0.0.0', port=5000, debug=False)

# 1. import library

In [ ]:
import os
import csv
import threading
from datetime import datetime
from typing import List, Optional, Dict
from flask import Flask, request, jsonify, send_from_directory
from flask_cors import CORS
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# 2. Path to save file

In [ ]:
# ==============================================
# 0. 路徑修正與 CSV 記錄工具
# ==============================================
USER_HOME = os.path.expanduser("~")
DESKTOP_PATH = os.path.join(USER_HOME, "Desktop/小花聊天紀錄")

if not os.path.exists(os.path.join(USER_HOME, "Desktop")):
    DESKTOP_PATH = os.path.join(USER_HOME, "小花聊天紀錄")

os.makedirs(DESKTOP_PATH, exist_ok=True)

class DailyCSVLogger:
    def __init__(self, basename: str, fieldnames: List[str]):
        self.basename = basename
        self.fieldnames = fieldnames
        self.lock = threading.Lock()

    def path_for_today(self) -> str:
        day = datetime.now().strftime("%Y-%m-%d")
        return os.path.join(DESKTOP_PATH, f"{self.basename}_{day}.csv")

    def append(self, row: Dict):
        path = self.path_for_today()
        with self.lock:
            file_exists = os.path.exists(path)
            with open(path, "a", newline="", encoding="utf-8-sig") as f:
                writer = csv.DictWriter(f, fieldnames=self.fieldnames, extrasaction="ignore")
                if not file_exists:
                    writer.writeheader()
                writer.writerow(row)

CHAT_FIELDS = ["對話時間", "使用者ID", "小花名稱", "設定性格", "當前情緒", "使用者提問", "AI回覆", "長者姓名", "長者年齡", "健康備註"]
CHAT_LOGGER = DailyCSVLogger("chatlogs", CHAT_FIELDS)

# 3. Pet Personality

In [ ]:
# ==============================================
# 1. 性格對照表 (對應你的 Swift Enum)
# ==============================================
PERSONALITY_PROMPTS = {
    "溫暖貼心": "你非常溫柔，說話非常有禮貌，常關心長輩的身體，會說『要記得喝水喔』，語氣充滿愛心，喜歡用❤️來表達情感。",
    "活潑調皮": "你充滿活力，說話節奏快，喜歡用驚嘆號，常說『快跟我玩！』或是『汪汪！我們去散步吧！』，喜歡用🤩。",
    "可愛粘人": "你很喜歡黏人，喜歡用很多很可愛的口頭禪例如汪汪、呀、喜歡用可愛的表情例如🥺，會想要纏著主人和你一起做不同的task。"
}

# 4. Download model

In [ ]:
# ==============================================
# 2. Flask & 模型
# ==============================================
app = Flask(__name__)
CORS(app)

model_name = "Qwen/Qwen2.5-1.5B-Instruct"
print("🚀 正在載入模型...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="auto")


emotion_map = {0: "Neutral", 1: "Happy", 2: "Sad", 3: "Surprise"}

# 5. API

In [ ]:
# ==============================================
# 3. API 接口
# ==============================================

# Download csv function 
@app.route('/download_logs')
def download_logs():
    try:
        day = datetime.now().strftime("%Y-%m-%d")
        filename = f"chatlogs_{day}.csv"
        return send_from_directory(DESKTOP_PATH, filename, as_attachment=True)
    except Exception as e:
        return f"下載失敗，可能是今天的檔案還沒產生：{str(e)}"

In [ ]:
@app.route('/ask_puppy', methods=['POST'])
def ask_puppy():
    try:
        data = request.json or {}
        
        user_msg = data.get('message', "")
        profile = data.get('user_profile') or {}
        p_name = profile.get('name', "未填寫")
        p_age = profile.get('age', "未填寫")
        p_notes = profile.get('health_notes', "無")
        
        user_id = data.get('user_id', "unknown")
        pet_name = data.get('pet_name', "小花")
        personality = data.get('personality', "溫暖貼心")
        e_idx = data.get('emotion_index', 0)
        current_emotion = emotion_map.get(int(e_idx), "Neutral")

        # 終端機顯示接收到的資料
        print("\n" + "📥" * 15)
        print(f"【收到提問】: {user_msg}")
        print(f"【長者資料】: 姓名:{p_name} | 年齡:{p_age} | 健康備註:{p_notes}")
        print("📥" * 15)

        # 💡 關鍵修正：明確的角色設定與性格載入
        detailed_personality = PERSONALITY_PROMPTS.get(personality, "你是一隻乖巧的小狗。")
        
        system_msg = (
            f"你現在是一隻名叫「{pet_name}」的虛擬寵物狗。"
            f"和你聊天的主人是一位長輩，名字叫「{p_name}」，今年{p_age}歲。"
            f"請注意主人的健康狀況：「{p_notes}」。"
            f"【你的性格設定】：{detailed_personality}"
            f"請完全以寵物狗的視角和口吻來回覆主人，字數盡量簡短可愛（大約30字），絕對不要承認自己是AI或人工智能。"
            f"主人現在的心情是：{current_emotion}。"
        )

        messages = [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_msg}
        ]
        
        # 生成 AI 回覆
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
        generated_ids = model.generate(**model_inputs, max_new_tokens=128, temperature=0.7)
        generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
        reply = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

        # 寫入 CSV
        CHAT_LOGGER.append({
            "對話時間": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "使用者ID": user_id, "小花名稱": pet_name, "設定性格": personality,
            "當前情緒": current_emotion, "使用者提問": user_msg, "AI回覆": reply,
            "長者姓名": p_name, "長者年齡": p_age, "健康備註": p_notes
        })

        # 終端機印出你要求的格式
        print(f"🌈 情緒: {current_emotion} | 🤖 回覆: {reply}\n")
        return jsonify({"reply": reply})

    except Exception as e:
        print(f"❌ 錯誤: {e}")
        return jsonify({"reply": "汪汪...小花頭有點暈，等我一下喔🥺"}), 200

if __name__ == '__main__':
    print("\n🐾 小花伺服器啟動中...")
    app.run(host='0.0.0.0', port=5000)